# Getting started with tsauditor

**Start here if this is your first time using the library.** Every other
notebook in `examples/` assumes you already know the basic workflow; this one
builds it from zero, on a small synthetic dataset, so you can run every cell
yourself and see real output.

What you'll do:
1. Build a tiny dataset with a planted leak and a planted sensor fault
2. Run `tsa.scan()` and read the report
3. Repair it with `apply_fixes()`
4. See a real, verified example of an LLM getting the API wrong, and what the
   correct usage actually is

Install: `pip install tsauditor`

## 1. Build a small dataset

300 trading days of a synthetic price series. `direction` is the target: did
the price go up **today**? `change_pct` is *today's* percentage price change,
computed the obvious way. Watch what happens when both exist in the same
frame, this is the exact bug class tsauditor was built for (it is a smaller
version of a real leak found in a Pakistani equity dataset, see
`examples/ogdc_leakage_case/` for the full story).

`volume` has two planted faults: a sensor-style stuck run (8 identical values
in a row) and a 5-day outage (missing values).

In [1]:
import numpy as np
import pandas as pd
import tsauditor as tsa

# report.summary() renders through `rich`, which auto-detects a Jupyter
# kernel and switches to fragmented HTML output (or, once that's turned
# off, ANSI color codes that a real terminal would render but a plain
# text stream won't). GitHub's notebook viewer strips or garbles both, so
# a committed notebook shows a wall of broken markup instead of a clean
# report. Forcing plain, uncoloured text here keeps this notebook's output
# identical whether it's run live in Jupyter or viewed statically on
# GitHub.
import tsauditor.report.summary as _tsa_report
from rich.console import Console as _Console


def _plain_console(*a, **kw):
    kw.setdefault("force_jupyter", False)
    kw.setdefault("force_terminal", False)
    kw.setdefault("no_color", True)
    return _Console(*a, **kw)


_tsa_report.Console = _plain_console

n = 300
idx = pd.bdate_range("2024-01-02", periods=n)

rng = np.random.default_rng(0)
price = pd.Series(100 + np.cumsum(rng.normal(0, 1, n)), index=idx)
change_pct = price.pct_change().fillna(0.0)
direction = (change_pct > 0).astype(float)  # today's move, the target

vrng = np.random.default_rng(0)  # independent stream
volume = vrng.integers(1000, 5000, n).astype(float)
volume[100:108] = volume[99]  # stuck sensor run
volume[150:155] = np.nan  # a 5-day outage

price_ma5 = price.rolling(5, min_periods=1).mean().shift(1)  # past-only

df = pd.DataFrame(
    {
        "price": price,
        "change_pct": change_pct,  # <- the leak: defines direction's sign
        "price_ma5": price_ma5,  # <- clean: uses only past data
        "volume": volume,  # <- has a stuck run and a gap
        "direction": direction,
    }
)
df.head()

,price,change_pct,price_ma5,volume,direction
2024-01-02,100.125730,0.000000,NaN,4402.0,0.0
2024-01-03,99.993625,-0.001319,100.125730,3547.0,0.0
2024-01-04,100.634048,0.006405,100.059678,3044.0,1.0
2024-01-05,100.738948,0.001042,100.251135,2079.0,1.0
2024-01-08,100.203279,-0.005317,100.373088,2231.0,0.0


## 2. Scan it

One call. `target=` matters: without it, every leakage check (LEK001-LEK005)
is **silently skipped**, no error, no warning, they simply never run. Forgetting
this is the single most common way to get a falsely clean report.

In [2]:
report = tsa.scan(df, target="direction", domain="finance", run_stationarity=False)
report.summary()

─────────────────────────────── tsauditor Report ───────────────────────────────



Dataset


  Rows       : 300


  Columns    : 5


  Time range : 2024-01-02 → 2025-02-24


...

  • ANO002 (change_pct): Review the point outliers in column 'change_pct' and 
decide whether to winsorize, transform, or treat them as data errors before 
modeling.


  • ANO003 (change_pct): Examine the local spikes in column 'change_pct': they 
deviate sharply from their neighbours and may be data-entry or data-feed errors.


  • ANO001 (volume): Investigate column 'volume' for a stuck sensor or a 
forward-filled gap: the value repeats unchanged for an unusually long run.


  • ANO001 (direction): Investigate column 'direction' for a stuck sensor or a 
forward-filled gap: the value repeats unchanged for an unusually long run.


  • PRF002 (volume): Handle the clustered missing values in column 'volume' 
(careful interpolation, limited forward-fill, or dropping the affected span) 
before modeling.


`scan()` returns a `GuardReport`, a real Python object with structured data,
not a block of printed text. Issues are bucketed by severity:

In [3]:
print("critical:", len(report.critical))
print("warnings:", len(report.warnings))
print("info    :", len(report.info))
print()
print("leaky_columns():", report.leaky_columns())

critical: 1
warnings: 5
info    : 0

leaky_columns(): ['change_pct']


`leaky_columns()` is the shortlist to review or remove before training. Notice
`price_ma5` is **not** in it, a rolling mean of *past* prices is a legitimate
feature and tsauditor stays quiet about it, even though it is built from the
same underlying price series as the leaky one. Only `change_pct`, which is
mathematically tied to the target's own sign, gets flagged.

Two other findings show up in the table above that are worth explaining rather
than ignoring, since a real report will have noise like this too:

- **`ANO002`/`ANO003` on `change_pct`**, a raw daily percentage-change series
  has some genuinely large-swing days by construction. These are separate,
  correct findings about volatility, not part of the leakage story.
- **`ANO001` on `direction` itself**, a binary 0/1 column naturally contains
  runs of repeated values (several up-days in a row). `ANO001` doesn't know or
  care that this column happens to be the target, it just reports what it
  sees. This is harmless here, but if it fired on a column you *do* intend to
  use as a feature, it would be worth a look.

## 3. Read a finding properly

Every `Issue` carries its reasoning, not just a verdict, `column`,
`description`, `evidence` (the numbers behind the decision), and `suggestion`.

In [4]:
leak = report.filter(code="LEK001")[0]

print("column     :", leak.column)
print("description:", leak.description)
print("evidence   :", leak.evidence)
print("suggestion :", leak.suggestion)

column     : change_pct
description: Feature 'change_pct' near-deterministically reproduces target 'direction' (auc score=1.0000 >= 0.95 for binary target). Likely data leakage — review before modeling.
evidence   : {'metric': 'auc', 'auc': 1.0, 'separation': 1.0, 'n_obs': 300, 'threshold': 0.95, 'target_type': 'binary'}
suggestion : Remove or reconstruct column 'change_pct': it near-deterministically reproduces the target variable and will leak. Keep it only if you can confirm it is genuinely available at prediction time.


`evidence['separation']` is the AUC-based separation score: 1.0 means the
feature perfectly determines the target's class, there is no ambiguity, this is
not a borderline call.

The sensor fault on `volume` shows up as its own, unrelated finding:

In [5]:
for i in report.filter(column="volume"):
    print(i.severity.upper(), i.code, "|", i.description)

WARNING ANO001 | Stuck values detected.
WARNING PRF002 | Column 'volume' contains clustered missing value sequences indicating an outage.


## 4. Repair it

`apply_fixes()` never touches your original `df`, it returns an independent
copy. The target column is never repaired (interpolating a 0/1 label would be
meaningless), and every change is logged.

In [6]:
clean = report.apply_fixes(df)

for f in report.last_fixes:
    print(f)

print()
print("NaNs before:", int(df["volume"].isna().sum()))
print("NaNs after :", int(clean["volume"].isna().sum()))
print("df untouched:", df["volume"].isna().sum() == 5)

{'column': 'change_pct', 'action': 'clip_outliers', 'cells_changed': 4, 'bounds': (-0.027021659254865493, 0.026729111772155367)}
{'column': 'change_pct', 'action': 'clip_spikes', 'cells_changed': 1}
{'column': 'volume', 'action': 'stuck_to_nan', 'cells_changed': 9, 'already_nan': 0}
{'column': 'volume', 'action': 'impute_interpolate', 'cells_changed': 14}

NaNs before: 5
NaNs after : 0
df untouched: True


`change_pct` is still there, `apply_fixes()` does not delete leaky columns
unless you explicitly ask it to (`leakage="drop"`), because dropping a feature
is a modeling decision, not something a data-quality tool should do silently.

In [7]:
clean_no_leak = report.apply_fixes(df, leakage="drop")
print('columns after leakage="drop":', list(clean_no_leak.columns))

columns after leakage="drop": ['price', 'price_ma5', 'volume', 'direction']


---

## 5. A real example of an LLM getting this wrong

This isn't hypothetical. Someone pasted a link to this repo into ChatGPT and
asked it to explain how to use `tsauditor`. Here is what it produced, and what
actually happens when you run it, verified against the real library, not
guessed.

**What ChatGPT wrote:**

```python
df = pd.DataFrame({
    "date": pd.date_range("2025-01-01", periods=10),
    "open": [...], "high": [...], "low": [...], "close": [...], "volume": [...]
})
df["target"] = (df["close"].shift(-1) > df["close"]).astype(int)
df["future_close"] = df["close"].shift(-1)

report = tsa.scan(df, target="target", domain="finance")
```

**Mistake #1: this raises, it does not scan.** `date` is a plain column, not
the index, and `time_col` was never passed. tsauditor deliberately refuses to
guess here, a numeric `RangeIndex` would otherwise be silently misread as
nanosecond timestamps near 1970.

In [8]:
df_bad = pd.DataFrame(
    {
        "date": pd.date_range("2025-01-01", periods=10),
        "close": [101, 102, 101, 103, 104, 105, 104, 106, 107, 108],
    }
)
df_bad["target"] = (df_bad["close"].shift(-1) > df_bad["close"]).astype(int)

try:
    tsa.scan(df_bad, target="target", domain="finance")
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

ValueError: DataFrame index is numeric, not datetime, and will not be coerced (it would be misread as epoch timestamps). Pass time_col='your_date_column' or set a DatetimeIndex before calling tsauditor.scan().


**The fix:** pass `time_col="date"` (or set the index yourself with
`df.set_index("date")` before calling `scan()`).

**Mistake #2:** ChatGPT then claimed `report.leaky_columns()` would return
`["future_close"]`. Even after fixing the index problem, it does not, because
the example only has **10 rows**, and every check in tsauditor requires
`min_obs=30` pairwise-complete observations by default before it will trust a
score enough to report anything. Below that, checks skip the column rather
than risk a spurious result from a handful of points.

In [9]:
df_fixed = df_bad.copy()
df_fixed["future_close"] = df_fixed["close"].shift(-1)

report_fixed = tsa.scan(df_fixed, target="target", domain="finance", time_col="date")
print("leaky_columns():", report_fixed.leaky_columns())
print('(empty, not ["future_close"], because 10 rows < min_obs=30)')

leaky_columns(): []
(empty, not ["future_close"], because 10 rows < min_obs=30)


**Mistake #3:** ChatGPT invented a `report.summary()` output, an ASCII checklist
with `✓`/`⚠`/`✗` symbols, and even labeled it *"illustrative"* rather than
admitting it had not run the code. The real format is a fixed-width report
block with a dataset header and severity counts, shown earlier in this notebook.
It never used checkmark symbols, and it groups by severity, not by check type.

**Mistake #4:** a suggested type hint used `-> AuditReport`. The real return
type is `GuardReport`. A class that does not exist cannot be imported, so code
written against that hint fails immediately.

**The pattern behind all of this:** an LLM without the actual source in
context will reach for what a *typical* Python data-quality library looks like
(dtypes, null counts, a friendly checklist print), not what this one actually
does (an object-based report, strict index handling, a `min_obs` floor against
spurious small-sample findings). None of the individual guesses were crazy,
they just weren't checked against the real package. The fix is the same one
every cell in this notebook follows: run it, and paste what actually happened.

---

## Where to go next

- [`examples/ogdc_leakage_case/`](../ogdc_leakage_case) — the real leak this
  library was built for, on real equity data, with the measured accuracy
  collapse once it's removed
- [`examples/sensor-example/`](../sensor-example) — structural and anomaly
  checks on a synthetic sensor stream, plus PDF report export
- [Quickstart](https://github.com/imann128/tsauditor/wiki/Quickstart) — every
  parameter used above, explained in full, including `available_at=` and
  `constraints=` for the two leakage checks that need explicit rules
- [API Reference](https://github.com/imann128/tsauditor/wiki/API-Reference) —
  every public function and parameter